# 04. 인덱싱 — ChromaDB `recipe_db_lab` (격리)

**무엇을 하나:** 벡터 + 레시피 정보를 ChromaDB 컬렉션에 넣는다. 한 레시피당 **3가지가 나란히** 저장된다:
- `embeddings` = bge-m3 벡터 (검색용)
- `documents` = embed_text (검색 텍스트 원문)
- `metadatas` = embed_text 를 **뺀 나머지 전부**(id·name·ingredients{분량·단위 포함}·nutrition·source...) — **임베딩 안 됨**, 검색 결과로 그대로 반환

**검색 동작:** 벡터로 의미가 가까운 걸 찾고 → 그 레시피의 **메타데이터를 꺼내** 보여준다. 그래서 분량·영양 같은 값은 벡터가 아니라 **메타에서** 나온다(05~06에서 직접 확인).

**격리:** 서빙 `recipe_db`(1,693건, 앱이 사용)를 건드리지 않으려고 **`recipe_db_lab`** 에 넣는다(학습/데모용). 운영 증분 주입(id upsert + embed_sig 가드)은 `s4_index.py` / `app.rag.indexer.upsert_recipes`.

In [ ]:
import sys,os,pickle
from pathlib import Path
B=(Path.cwd().parent/'backend') if Path.cwd().name=='notebooks' else Path.cwd()/'backend'
B=B.resolve(); sys.path.insert(0,str(B)); os.chdir(B)
from dotenv import load_dotenv; load_dotenv()
from app.rag.embedder import Embedder
from app.rag.indexer import get_chroma_client,_sanitize_metadata
d=pickle.loads((B/'data'/'lab'/'03_vecs.pkl').read_bytes()); recs,vecs=d['recs'],d['vecs']
sig=Embedder().signature
c=get_chroma_client()
try: c.delete_collection('recipe_db_lab')
except Exception: pass
col=c.create_collection('recipe_db_lab',metadata={'hnsw:space':'cosine','embed_sig':sig})
col.add(ids=[r['id'] for r in recs],documents=[r['embed_text'] for r in recs],embeddings=vecs,metadatas=[_sanitize_metadata(r) for r in recs])
print('recipe_db_lab indexed:',col.count(),'sig:',sig)